<a href="https://colab.research.google.com/github/lawrennd/qig-code/blob/main/examples/origin_two_qutrit_worked_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Worked Example: Two Qutrits at the LME Origin

This notebook implements the worked example described in *The Origin of the Inaccessible Game* for a **two-qutrit** system.

We will:

- Construct the LME (locally maximally entangled) origin state $|\Phi_3\rangle$.
- Verify **global purity** and **maximally mixed marginals**.
- Compute joint and marginal von Neumann entropies and the multi-information $I$.
- Demonstrate **marginal-preserving directions**:
  - local-unitary (gauge) commutator flows
  - an explicit ``correlation-only'' tangent perturbation with vanishing partial traces
- Compute the **entropy-time interval** implied by $\tfrac{\text{d}\mathsf{H}}{\text{d}t}=\log 2$.

Optional: a short extension to **four qutrits** (two entangled pairs) for multi-pair context.

In [ ]:
# Auto-install QIG package if not available (Colab-friendly)
import os

try:
    import qig  # noqa: F401
except ImportError:
    print("Installing QIG package...")
    %pip install -q git+https://github.com/lawrennd/qig-code.git
    import qig  # noqa: F401
    print("✓ QIG package installed")

In [ ]:
import numpy as np
from scipy.linalg import expm

from qig.core import create_lme_state, partial_trace, von_neumann_entropy, marginal_entropies
from qig.pair_operators import gell_mann_generators

np.set_printoptions(precision=6, suppress=True)

In [ ]:
# Two qutrits
n_sites = 2
d = 3
dims = [d] * n_sites
D = d**n_sites

rho, dims = create_lme_state(n_sites=n_sites, d=d)

S_joint = von_neumann_entropy(rho)
h = marginal_entropies(rho, dims)
C = float(np.sum(h))
I = C - S_joint

print(f"dims = {dims}, D = {D}")
print(f"Tr(rho) = {np.trace(rho):.6f}")
print(f"Purity Tr(rho^2) = {np.trace(rho @ rho).real:.6f}")
print(f"S(rho) = {S_joint:.6f}")
print(f"marginal entropies h_i = {h}")
print(f"C = sum h_i = {C:.6f}")
print(f"multi-information I = C - S = {I:.6f}")

# Verify reduced states are maximally mixed
rho0 = partial_trace(rho, dims, keep=0)
rho1 = partial_trace(rho, dims, keep=1)
print("\nReduced states:")
print("rho_0:\n", rho0)
print("rho_1:\n", rho1)
print("||rho_i - I/d||_F:", np.linalg.norm(rho0 - np.eye(d)/d), np.linalg.norm(rho1 - np.eye(d)/d))

In [ ]:
# Marginal-preserving (gauge) directions: local-unitary commutator flow
# Pick a simple su(3) generator and lift it to site 0: K = lambda ⊗ I
lambdas = gell_mann_generators(d)
lam3 = lambdas[2]  # analogous to diagonal generator
K_local = np.kron(lam3, np.eye(d))

alpha = 0.37
U = expm(-1j * alpha * K_local)
rho_U = U @ rho @ U.conj().T

h_U = marginal_entropies(rho_U, dims)
rho0_U = partial_trace(rho_U, dims, keep=0)
rho1_U = partial_trace(rho_U, dims, keep=1)

print("Local-unitary test:")
print("Δh =", h_U - h)
print("||rho0_U - rho0||_F:", np.linalg.norm(rho0_U - rho0))
print("||rho1_U - rho1||_F:", np.linalg.norm(rho1_U - rho1))

# First-order version: dot{rho} = -i[K, rho]
dr = -1j * (K_local @ rho - rho @ K_local)
print("\nFirst-order (commutator) test:")
print("||Tr_2(dr)||_F:", np.linalg.norm(partial_trace(dr, dims, keep=0)))
print("||Tr_1(dr)||_F:", np.linalg.norm(partial_trace(dr, dims, keep=1)))

In [ ]:
# Correlation-only tangent perturbations: build a Hermitian, trace-zero matrix with vanishing partial traces
# For bipartite (d×d) systems, a convenient projector is:
#   H_corr = H - Tr_2(H) ⊗ I/d - I/d ⊗ Tr_1(H) + Tr(H) I⊗I / d^2

rng = np.random.default_rng(0)
X = rng.normal(size=(D, D)) + 1j * rng.normal(size=(D, D))
H = 0.5 * (X + X.conj().T)
H = H - (np.trace(H) / D) * np.eye(D)  # traceless

A = partial_trace(H, dims, keep=0)  # Tr_2(H)
B = partial_trace(H, dims, keep=1)  # Tr_1(H)

H_corr = H - np.kron(A, np.eye(d) / d) - np.kron(np.eye(d) / d, B) + (np.trace(H) / (d**2)) * np.eye(D)

print("Checks for correlation-only tangent H_corr:")
print("Tr(H_corr) =", np.trace(H_corr))
print("||Tr_2(H_corr)||_F:", np.linalg.norm(partial_trace(H_corr, dims, keep=0)))
print("||Tr_1(H_corr)||_F:", np.linalg.norm(partial_trace(H_corr, dims, keep=1)))

# H_corr is a tangent direction; it is not guaranteed to keep rho+εH_corr PSD for finite ε.
eps = 1e-4
rho_eps = rho + eps * H_corr

# Reduced-state change is first-order zero by construction
rho0_eps = partial_trace(rho_eps, dims, keep=0)
rho1_eps = partial_trace(rho_eps, dims, keep=1)
print("\nFirst-order marginal change check (should scale ~eps * 0):")
print("||rho0_eps - rho0||_F:", np.linalg.norm(rho0_eps - rho0))
print("||rho1_eps - rho1||_F:", np.linalg.norm(rho1_eps - rho1))

In [ ]:
# Entropy-time interval
# If dS/dt_entropy = log 2, then going from S=0 to S=log D takes time log(D)/log(2) = log2(D)
S_end = np.log(D)  # maximally mixed on D-dimensional space
entropy_time_interval = S_end / np.log(2)

print(f"Maximal joint entropy log(D) = {S_end:.6f}")
print(f"Entropy-time interval to reach S=log(D): Δt_entropy = log(D)/log 2 = {entropy_time_interval:.6f} = log2(D)")
print(f"For two qutrits, log2(D)=log2(9)={np.log2(D):.6f} = 2 log2(3) = {2*np.log2(3):.6f}")

## Optional: Four Qutrits (Two Entangled Pairs)

If you want to connect directly to the multi-pair regularisation machinery, set `n_sites=4` (equivalently `n_pairs=2`). In `qig.core.create_lme_state`, the LME origin for even `n_sites` is a **product of maximally entangled pairs**.

For a full multi-pair discussion (regularisation choices, ``many meridians, one pole''), see `examples/multi_pair_regularisation.ipynb`.

In [ ]:
# Four qutrits (two maximally entangled pairs)
n_sites4 = 4
d4 = 3
dims4 = [d4] * n_sites4
D4 = d4**n_sites4

rho4, dims4 = create_lme_state(n_sites=n_sites4, d=d4)
S4 = von_neumann_entropy(rho4)
h4 = marginal_entropies(rho4, dims4)
C4 = float(np.sum(h4))
I4 = C4 - S4

print(f"dims4={dims4}, D4={D4}")
print(f"Purity Tr(rho4^2) = {np.trace(rho4 @ rho4).real:.6f}")
print(f"S(rho4) = {S4:.6f}")
print(f"marginal entropies h_i = {h4}")
print(f"C4 = sum h_i = {C4:.6f} (expected {n_sites4*np.log(d4):.6f})")
print(f"multi-information I4 = {I4:.6f}")

# Entropy-time interval to reach maximally mixed on D4
print(f"Entropy-time interval to reach S=log(D4): log2(D4) = {np.log2(D4):.6f}")